# Fundamentos de una Red Neuronal Simple

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Asignatura Ministerial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

El objetivo de este cuaderno es construir y entrenar una red neuronal mínima (de una sola neurona) para aprender la relación lineal entre temperaturas en grados Celsius y Fahrenheit. Esto les permitirá comprender de manera práctica conceptos básicos como capas, pesos, sesgos y el proceso de optimización antes de avanzar hacia problemas complejos de procesamiento de imágenes.

## Resultados de aprendizaje

Al final de este notebook van a poder:
1. Distinguir entre variables de entrada y variables de salida en un modelo de aprendizaje supervisado.
2. Definir una arquitectura secuencial con una sola capa y una neurona en TensorFlow.
3. Configurar y compilar un modelo seleccionando una función de pérdida y un optimizador.
4. Entrenar el modelo, analizar la curva de pérdida para verificar el aprendizaje y realizar predicciones.
5. Interpretar los pesos y el sesgo aprendidos por la neurona en comparación con la fórmula física real.

## Terminología clave (Microglosario)

*   **Entrada (Input):** El dato que ingresa a la red neuronal (en este caso, una temperatura en grados Celsius).
*   **Salida (Output):** El valor que el modelo intenta predecir (en este caso, la temperatura en grados Fahrenheit).
*   **Capa Densa (Dense):** Tipo de capa donde cada neurona está conectada a todas las entradas de la etapa anterior.
*   **Época (Epoch):** Una pasada completa de la red por todo el conjunto de datos durante el entrenamiento.
*   **Pérdida (Loss):** Métrica que mide qué tan alejadas están las predicciones del modelo respecto de los valores reales.
*   **Pesos y Sesgo (Weights & Bias):** Parámetros internos ajustables que la red modifica mediante optimización para aprender la relación matemática entre variables.

## 1. Configuración del Entorno y Carga de Datos

Comencemos importando las herramientas de análisis de datos y modelado. Para simplificar el acceso a los datos en entornos de desarrollo locales o remotos, utilizaremos un archivo local que prepararon previamente.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Opcional: Configuración para detectar el entorno de ejecución
try:
    from google.colab import drive
    print("✦ Entorno detectado: Google Colab")
except ImportError:
    print("✦ Entorno detectado: Local o Servidor Jupyter")

print("✓ Librerías listas y semilla lista para configurar.")

In [ ]:
# Definimos la ruta relativa al archivo de datos
ruta_datos = "./datos/celsius.csv"

# Cargamos la tabla utilizando Pandas
datos_temperatura = pd.read_csv(ruta_datos)

# Mostramos las primeras filas para comprender el formato
display(datos_temperatura.head())
print(f"✓ Cantidad de muestras cargadas: {len(datos_temperatura)}")

## Consigna de Lectura e Interpretación

Observen la tabla impresa anteriormente. ¿Cuál de las dos columnas representa la variable de entrada física que queremos transformar y cuál la variable de salida objetivo?

## 2. Separación de Variables de Entrada y Salida

Para que la red neuronal pueda aprender de forma supervisada, debemos proveerle las variables de entrada y sus respectivas respuestas deseadas de manera independiente.

In [ ]:
# Seleccionamos la columna de Celsius para las entradas
entradas_celsius = datos_temperatura["celsius"].values

# Seleccionamos la columna de Fahrenheit para las salidas esperadas
salidas_fahrenheit = datos_temperatura["fahrenheit"].values

print("✓ Vectores cargados:")
print("  • Entradas (X):", entradas_celsius)
print("  • Salidas esperadas (y):", salidas_fahrenheit)

## 3. Construcción de la Red Neuronal Mínima

Construiremos un modelo secuencial utilizando la API de Keras. Dado que el cambio físico de Celsius a Fahrenheit es puramente lineal, nos basta con una única neurona sin funciones de activación complejas.

In [ ]:
# Diseñamos la arquitectura secuencial
modelo = tf.keras.Sequential([
    # Capa de Entrada: recibe una única temperatura
    tf.keras.layers.Input(shape=(1,)),
    
    # Capa de Salida: una única neurona densa
    tf.keras.layers.Dense(units=1)
])

# Compilamos especificando el optimizador Adam y la función de pérdida
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss="mean_squared_error"
)

print("✓ Red neuronal configurada e inicializada.")
modelo.summary()

## Consigna de Lectura e Interpretación

Revisen el bloque de `summary()`. ¿Cuántos parámetros ajustables (pesos y sesgos) posee este modelo? ¿Por qué tiene esa cantidad exacta?

## 4. Entrenamiento del Modelo

Procederemos al entrenamiento de la red por un total de 800 épocas. En este punto, la neurona intentará ajustar gradualmente sus parámetros internos mediante retropropagación (backpropagation) para minimizar la pérdida.

In [ ]:
print("✦ Iniciando el proceso de optimización del modelo...")

# Entrenamos el modelo y guardamos el historial del error
entrenamiento = modelo.fit(
    entradas_celsius,
    salidas_fahrenheit,
    epochs=800,
    verbose=0
)

perdida_final = entrenamiento.history["loss"][-1]
print(f"✓ Proceso completado.")
print(f"  • Pérdida (MSE) final de entrenamiento: {perdida_final:.4f}")

## 5. Visualización del Progreso del Aprendizaje

Graficaremos el comportamiento del error o pérdida con respecto a cada época transcurrida. Esto nos permitirá diagnosticar si el modelo efectivamente aprendió de los ejemplos.

In [ ]:
# Graficamos la curva de pérdida
plt.figure(figsize=(8, 4))
plt.plot(entrenamiento.history["loss"], color="darkblue", linewidth=2)
plt.title("Curva de Pérdida durante el Entrenamiento", fontsize=12, fontweight="bold", pad=15)
plt.xlabel("Época", fontsize=10)
plt.ylabel("Pérdida (MSE)", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Consigna de Lectura e Interpretación

Analicen la pendiente de la curva de error. ¿En qué época aproximada observan que la pérdida se estabiliza y deja de decrecer? ¿Qué significa este comportamiento?

## 6. Inferencia del Modelo en un Dato Nuevo

Ahora comprobaremos la capacidad de predicción (inferencia) de nuestro modelo utilizando una temperatura Celsius nueva, la cual no estaba presente en la tabla original de entrenamiento.

In [ ]:
temperatura_consulta = 12

# Preparamos el formato de entrada esperado por el modelo
entrada_numpy = np.array([[temperatura_consulta]])

# Realizamos la predicción de salida
resultado_prediccion = modelo.predict(entrada_numpy, verbose=0)
valor_fahrenheit = resultado_prediccion[0][0]

print(f"✦ Temperatura de entrada: {temperatura_consulta} °C")
print(f"✓ Equivalencia estimada por la red: {valor_fahrenheit:.2f} °F")

## 7. Análisis e Interpretación Matemática de los Pesos Aprendidos

Dado que la relación lineal física entre ambas escalas está gobernada por la fórmula:

$$\text{Fahrenheit} = \text{Celsius} \times 1.8 + 32$$

Nuestra neurona (que implementa la ecuación básica $y = x \cdot w + b$) debió aproximar el peso $w$ a $1.8$ y el sesgo o bias $b$ a $32$.

In [ ]:
# Extraemos los parámetros de la única neurona densa
peso_final, sesgo_final = modelo.layers[0].get_weights()

print("✓ Pesos definitivos obtenidos por el modelo:")
print("  • Peso aprendido (w):", peso_final[0][0])
print("  • Sesgo aprendido (b):", sesgo_final[0])

### Preguntas para pensar:
1. ¿Qué tan cerca quedaron el peso y el sesgo estimados por la neurona respecto de los coeficientes de la fórmula física real ($1.8$ y $32$)?
2. ¿Por qué creen que el modelo no llegó con precisión milimétrica absoluta a esos números exactos? (Pistas: piensen en la cantidad de ejemplos del dataset, el número de épocas y la tasa de aprendizaje del optimizador).

---

## Cierre y Siguientes Pasos

Comprobaron de forma directa cómo una red neuronal mínima puede auto-ajustar sus pesos internos mediante descenso de gradiente para modelar con precisión una relación lineal real.

En los siguientes cuadernos daremos el salto de estas escalas numéricas unidimensionales hacia las imágenes complejas. Allí las entradas ya no serán coeficientes numéricos únicos, sino mapas estructurados de píxeles organizados en canales de color.